In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import zipfile

# SETTINGS
DAILY_BUDGET = 300.00
CHANNELS = ["Search", "Social", "Display"]
LOOKBACK_DAYS = 7
MIN_DAYS = 3

# 1. UPLOAD THE DATASET
print("Upload your advertising Excel or CSV file.")
uploaded = files.upload()
filename = next(iter(uploaded))

if filename.lower().endswith(".xlsx"):
    data = pd.read_excel(filename)
else:
    data = pd.read_csv(filename)

# 2. CHECK AND PREPARE THE DATA
data.columns = data.columns.str.strip().str.lower()

required = [
    "date", "channel", "spend",
    "impressions", "clicks", "conversions"
]

if not set(required).issubset(data.columns):
    raise ValueError(
        "Your file must contain: " + ", ".join(required)
    )

data = data[required].copy()
data["date"] = pd.to_datetime(
    data["date"], errors="raise"
).dt.normalize()

data["channel"] = (
    data["channel"].astype(str).str.strip().str.title()
)

if data["date"].isna().any():
    raise ValueError("The file contains missing dates.")

if not data["channel"].isin(CHANNELS).all():
    raise ValueError("Channels must be Search, Social, or Display.")

for column in required[2:]:
    data[column] = pd.to_numeric(data[column], errors="raise")

    if (
        not np.isfinite(data[column]).all()
        or (data[column] < 0).any()
    ):
        raise ValueError("Metrics must be nonnegative numbers.")

if (
    (data["clicks"] > data["impressions"]).any()
    or (data["conversions"] > data["clicks"]).any()
):
    raise ValueError(
        "Expected conversions <= clicks <= impressions."
    )

if data.duplicated(["date", "channel"]).any():
    raise ValueError("There are duplicate date/channel rows.")

if not np.isfinite(DAILY_BUDGET) or DAILY_BUDGET < 0.03:
    raise ValueError("Enter a daily budget of at least $0.03.")

data = data.sort_values(["date", "channel"])

# Save a CSV version of the uploaded dataset.
data.to_csv("advertising_data.csv", index=False)

# 3. CALCULATE CHANNEL METRICS
def calculate_metrics(frame):
    summary = (
        frame.groupby("channel")[
            ["spend", "impressions", "clicks", "conversions"]
        ]
        .sum()
        .reindex(CHANNELS, fill_value=0)
    )

    summary["CTR (%)"] = (
        summary["clicks"]
        / summary["impressions"].replace(0, np.nan)
        * 100
    )

    summary["Conversion Rate (%)"] = (
        summary["conversions"]
        / summary["clicks"].replace(0, np.nan)
        * 100
    )

    summary["CPA ($)"] = (
        summary["spend"]
        / summary["conversions"].replace(0, np.nan)
    )

    return summary


historical_metrics = calculate_metrics(data)

print("\nHISTORICAL CHANNEL PERFORMANCE")
display(historical_metrics.round(2))

# 4. START WITH AN EQUAL DAILY BUDGET
# Use whole cents so rounding cannot break the budget limits.
total_cents = int(round(DAILY_BUDGET * 100))
equal_amount, leftover = divmod(total_cents, 3)

previous = np.array([
    equal_amount + (i < leftover)
    for i in range(3)
], dtype=int)

decision_log = []

# 5. ANALYZE EACH DAY AND RECOMMEND THE NEXT DAY'S BUDGET
for day in sorted(data["date"].unique()):
    day = pd.Timestamp(day)

    # Only use information available through this day.
    window_start = day - pd.Timedelta(days=LOOKBACK_DAYS - 1)

    recent = data[
        (data["date"] >= window_start)
        & (data["date"] <= day)
    ]

    metrics = calculate_metrics(recent)

    days_available = (
        recent.groupby("channel")["date"]
        .nunique()
        .reindex(CHANNELS, fill_value=0)
    )

    enough_data = (
        (days_available >= MIN_DAYS).all()
        and (metrics["spend"] > 0).all()
        and (metrics["clicks"] > 0).all()
    )

    # Conversions per dollar = inverse CPA.
    efficiency = (
        metrics["conversions"]
        / metrics["spend"].replace(0, np.nan)
    ).fillna(0)

    # Each channel gets at least 10% of the total budget.
    # Changes cannot exceed 20% of its previous budget.
    lower = np.maximum(
        np.ceil(previous * 0.80),
        np.ceil(total_cents * 0.10)
    ).astype(int)

    upper = np.floor(previous * 1.20).astype(int)

    if not enough_data or efficiency.sum() == 0:
        recommended = previous.copy()

        explanation = (
            "Kept the previous allocation because there "
            "was insufficient data or no conversions."
        )

    else:
        recommended = lower.copy()
        remaining = total_cents - recommended.sum()

        # Give remaining funds to the most efficient channels,
        # stopping at each channel's permitted upper limit.
        ranking = np.argsort(
            -efficiency.to_numpy(),
            kind="stable"
        )

        for i in ranking:
            available_room = upper[i] - recommended[i]
            increase = min(remaining, available_room)

            recommended[i] += increase
            remaining -= increase

        if remaining != 0:
            raise ValueError(
                "Could not allocate the budget within the limits."
            )

        explanation = (
            "Prioritized channels with more conversions per "
            "dollar while enforcing the 20% daily change limit "
            "and 10% minimum allocation."
        )

    # 6. VERIFY THE BUDGET RULES
    assert recommended.sum() == total_cents
    assert (recommended >= lower).all()
    assert (recommended <= upper).all()

    # 7. LOG EACH CHANNEL'S DECISION
    for i, channel in enumerate(CHANNELS):
        old_budget = previous[i] / 100
        new_budget = recommended[i] / 100

        if new_budget > old_budget:
            action = "Increase"
        elif new_budget < old_budget:
            action = "Decrease"
        else:
            action = "Hold"

        cpa = metrics.loc[channel, "CPA ($)"]

        cpa_text = (
            f"${cpa:.2f}"
            if pd.notna(cpa)
            else "undefined because conversions were zero"
        )

        decision_log.append({
            "Data Through": day.strftime("%Y-%m-%d"),
            "Recommendation Date": (
                day + pd.Timedelta(days=1)
            ).strftime("%Y-%m-%d"),
            "Channel": channel,
            "Previous Budget": old_budget,
            "Recommended Budget": new_budget,
            "Change (%)": (
                (new_budget - old_budget) / old_budget * 100
            ),
            "CTR (%)": metrics.loc[channel, "CTR (%)"],
            "Conversion Rate (%)": metrics.loc[
                channel, "Conversion Rate (%)"
            ],
            "CPA ($)": cpa,
            "Action": action,
            "Reason": (
                f"{action}. Recent CPA: {cpa_text}. "
                + explanation
            )
        })

    # Simulation assumption: the last recommendation becomes
    # the previous allocation for the next decision.
    previous = recommended.copy()

# 8. SHOW THE FINAL RECOMMENDATION
log = pd.DataFrame(decision_log)

latest_date = log["Recommendation Date"].max()

next_day_budget = log[
    log["Recommendation Date"] == latest_date
].copy()

print("\nNEXT-DAY BUDGET RECOMMENDATION")
display(
    next_day_budget[
        [
            "Recommendation Date",
            "Channel",
            "Previous Budget",
            "Recommended Budget",
            "Change (%)",
            "Action"
        ]
    ].round(2)
)

print("\nREASONS")
for _, row in next_day_budget.iterrows():
    print(f"\n{row['Channel']}: {row['Reason']}")

print("\nAll budget checks passed.")
print(
    "Total recommended daily budget:",
    f"${next_day_budget['Recommended Budget'].sum():.2f}"
)

print(
    "\nThese are simulated recommendations, "
    "not actual results from spending the new budgets."
)

# 9. SAVE AND DOWNLOAD THE OUTPUTS
historical_metrics.to_csv("channel_metrics.csv")
log.to_csv("decision_log.csv", index=False)
next_day_budget.to_csv("next_day_budget.csv", index=False)

output_files = [
    "advertising_data.csv",
    "channel_metrics.csv",
    "decision_log.csv",
    "next_day_budget.csv"
]

with zipfile.ZipFile(
    "Member3_Results.zip", "w", zipfile.ZIP_DEFLATED
) as archive:
    for output in output_files:
        archive.write(output)

files.download("Member3_Results.zip")

Upload your advertising Excel or CSV file.


Saving BUS118S as_optimization_21_day_dataset.xlsx to BUS118S as_optimization_21_day_dataset.xlsx

HISTORICAL CHANNEL PERFORMANCE


,spend,impressions,clicks,conversions,CTR (%),Conversion Rate (%),CPA ($)
channel,,,,,,,
Search,2154.58,119960,6348,528,5.29,8.32,4.08
Social,2047.80,187356,4782,196,2.55,4.10,10.45
Display,2090.31,347639,2977,71,0.86,2.38,29.44



NEXT-DAY BUDGET RECOMMENDATION


,Recommendation Date,Channel,Previous Budget,Recommended Budget,Change (%),Action
60,2026-10-22,Search,240.0,240.0,0.0,Hold
61,2026-10-22,Social,30.0,30.0,0.0,Hold
62,2026-10-22,Display,30.0,30.0,0.0,Hold



REASONS

Search: Hold. Recent CPA: $4.17. Prioritized channels with more conversions per dollar while enforcing the 20% daily change limit and 10% minimum allocation.

Social: Hold. Recent CPA: $10.18. Prioritized channels with more conversions per dollar while enforcing the 20% daily change limit and 10% minimum allocation.

Display: Hold. Recent CPA: $25.70. Prioritized channels with more conversions per dollar while enforcing the 20% daily change limit and 10% minimum allocation.

All budget checks passed.
Total recommended daily budget: $300.00

These are simulated recommendations, not actual results from spending the new budgets.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>